In [ ]:
# NotY3dGenAI - Professional 3D Model Generator
# Simplified working version

import os
import sys
import time
import torch
import numpy as np
from PIL import Image, ImageDraw
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import drive, files
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

print("📦 Installing required packages...")
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install -q transformers accelerate diffusers
%pip install -q opencv-python-headless
%pip install -q trimesh
%pip install -q scikit-learn
%pip install -q plotly ipywidgets

# Import required libraries
import cv2
from pathlib import Path
import trimesh

print("✅ All packages installed!")

# Create directories
Path("/content/noty3d_models").mkdir(exist_ok=True)
Path("/content/drive/MyDrive/NotY3D_Models").mkdir(exist_ok=True)

class Simple3DGenerator:
    """Simple but effective 3D model generator"""
    
    def __init__(self):
        self.device = self.setup_device()
        
    def setup_device(self):
        if torch.cuda.is_available():
            print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
            return "cuda"
        return "cpu"
    
    def create_human_mesh(self, prompt, detail_level=1.0):
        """Create a detailed human mesh based on prompt"""
        
        # Parse prompt for characteristics
        prompt_lower = prompt.lower()
        
        # Determine body type
        if "muscular" in prompt_lower or "strong" in prompt_lower:
            body_scale = 1.2
        elif "slim" in prompt_lower or "thin" in prompt_lower:
            body_scale = 0.8
        else:
            body_scale = 1.0
        
        # Create parametric human model
        vertices = []
        faces = []
        
        # Parameters
        height = 2.0 * detail_level
        width = 0.8 * body_scale
        depth = 0.4 * body_scale
        
        # Create body parts
        # Head (sphere)
        head_center = [0, height * 0.8, 0]
        head_radius = 0.25 * detail_level
        
        # Torso (cylinder-like)
        torso_bottom = height * 0.3
        torso_top = height * 0.7
        torso_width = width
        torso_depth = depth
        
        # Generate vertices
        # Head vertices
        for theta in np.linspace(0, np.pi, 12):
            for phi in np.linspace(0, 2*np.pi, 24):
                x = head_center[0] + head_radius * np.sin(theta) * np.cos(phi)
                y = head_center[1] + head_radius * np.cos(theta)
                z = head_center[2] + head_radius * np.sin(theta) * np.sin(phi)
                vertices.append([x, y, z])
        
        # Torso vertices
        torso_layers = 20
        for i in range(torso_layers):
            t = i / (torso_layers - 1)
            y = torso_bottom + t * (torso_top - torso_bottom)
            # Taper the waist
            waist_factor = 1 - 0.3 * np.sin(np.pi * t)
            r_x = torso_width * waist_factor / 2
            r_z = torso_depth * waist_factor / 2
            
            for angle in np.linspace(0, 2*np.pi, 24):
                x = r_x * np.cos(angle)
                z = r_z * np.sin(angle)
                vertices.append([x, y, z])
        
        # Arm vertices (left and right)
        arm_length = height * 0.4
        arm_radius = 0.12
        
        for side in [-1, 1]:  # -1 left, 1 right
            shoulder_y = height * 0.7
            # Upper arm
            for i in range(10):
                t = i / 9
                y = shoulder_y - t * arm_length * 0.5
                r = arm_radius * (1 - t * 0.3)
                for angle in np.linspace(0, 2*np.pi, 12):
                    x = side * (torso_width/2 + 0.05) + r * np.cos(angle)
                    z = r * np.sin(angle)
                    vertices.append([x, y, z])
            
            # Lower arm
            elbow_y = shoulder_y - arm_length * 0.5
            for i in range(10):
                t = i / 9
                y = elbow_y - t * arm_length * 0.5
                r = arm_radius * (1 - t * 0.2)
                for angle in np.linspace(0, 2*np.pi, 12):
                    x = side * (torso_width/2 + 0.05) + r * np.cos(angle)
                    z = r * np.sin(angle)
                    vertices.append([x, y, z])
        
        # Leg vertices
        leg_length = height * 0.5
        leg_radius = 0.15
        
        for side in [-1, 1]:
            hip_y = height * 0.3
            # Upper leg
            for i in range(10):
                t = i / 9
                y = hip_y - t * leg_length * 0.5
                r = leg_radius * (1 - t * 0.2)
                for angle in np.linspace(0, 2*np.pi, 12):
                    x = side * (torso_width/3) + r * np.cos(angle)
                    z = r * np.sin(angle)
                    vertices.append([x, y, z])
            
            # Lower leg
            knee_y = hip_y - leg_length * 0.5
            for i in range(10):
                t = i / 9
                y = knee_y - t * leg_length * 0.5
                r = leg_radius * (1 - t * 0.3)
                for angle in np.linspace(0, 2*np.pi, 12):
                    x = side * (torso_width/3) + r * np.cos(angle)
                    z = r * np.sin(angle)
                    vertices.append([x, y, z])
        
        # Create faces (connect vertices)
        vertices = np.array(vertices)
        
        # Simple triangulation based on vertex proximity
        from scipy.spatial import Delaunay
        
        # Use only a subset for Delaunay to avoid memory issues
        sample_size = min(500, len(vertices))
        indices = np.random.choice(len(vertices), sample_size, replace=False)
        sample_vertices = vertices[indices]
        
        try:
            tri = Delaunay(sample_vertices[:, [0, 2]])  # Use XZ plane
            for simplex in tri.simplices:
                if len(simplex) == 3:
                    faces.append([indices[simplex[0]], indices[simplex[1]], indices[simplex[2]]])
        except:
            # Fallback: create simple grid faces
            pass
        
        # Create mesh
        mesh = trimesh.Trimesh(vertices=vertices, faces=np.array(faces) if faces else [])
        
        # Smooth mesh
        if detail_level > 0.5:
            mesh = mesh.smooth(iterations=3)
        
        return mesh
    
    def create_authoritative_texture(self, prompt, size=1024):
        """Create a texture that matches the authoritative tone"""
        
        img = Image.new('RGB', (size, size), color=(20, 20, 30))
        draw = ImageDraw.Draw(img)
        
        # Create authoritative color scheme (dark, powerful colors)
        colors = [(40, 40, 60), (50, 50, 70), (60, 60, 80), (30, 30, 50)]
        
        # Create gradient pattern
        for i in range(size):
            intensity = int(30 + 20 * np.sin(i * 0.01))
            color = (intensity, intensity, intensity + 20)
            draw.line([(0, i), (size, i)], fill=color)
        
        # Add authoritative geometric patterns
        center = size // 2
        for r in range(50, min(size//2, 300), 50):
            draw.ellipse([center-r, center-r, center+r, center+r], 
                        outline=(80, 80, 100), width=3)
        
        # Add diagonal lines for power
        draw.line([(0, 0), (size, size)], fill=(100, 100, 130), width=5)
        draw.line([(size, 0), (0, size)], fill=(100, 100, 130), width=5)
        
        return img

class NotY3dGenAI:
    """Main application"""
    
    def __init__(self):
        self.generator = Simple3DGenerator()
        self.current_mesh = None
        self.current_model_path = None
        self.create_ui()
    
    def create_ui(self):
        """Create professional UI"""
        
        display(HTML("""
        <style>
            @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
            
            * { font-family: 'Inter', sans-serif; }
            
            .title {
                font-size: 52px;
                font-weight: 800;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
                -webkit-background-clip: text;
                -webkit-text-fill-color: transparent;
                text-align: center;
                margin-bottom: 10px;
            }
            
            .subtitle {
                text-align: center;
                color: #888;
                margin-bottom: 30px;
            }
            
            .info {
                background: linear-gradient(135deg, #667eea, #764ba2);
                color: white;
                padding: 20px;
                border-radius: 15px;
                margin: 20px 0;
            }
            
            .btn-generate {
                background: linear-gradient(135deg, #667eea, #764ba2);
                color: white;
                border: none;
                padding: 15px;
                font-size: 18px;
                font-weight: 600;
                border-radius: 10px;
                cursor: pointer;
                width: 100%;
                transition: transform 0.2s;
            }
            
            .btn-generate:hover { transform: translateY(-2px); }
            
            .btn-download {
                background: linear-gradient(135deg, #11998e, #38ef7d);
                color: white;
                border: none;
                padding: 12px;
                font-size: 16px;
                font-weight: 600;
                border-radius: 8px;
                cursor: pointer;
                width: 100%;
                margin-top: 10px;
            }
            
            .btn-download:hover { transform: translateY(-2px); }
            
            .status {
                background: #2d2d2d;
                color: #0f0;
                padding: 10px;
                border-radius: 5px;
                margin-top: 10px;
                white-space: pre-wrap;
            }
        </style>
        """))
        
        display(HTML('<div class="title">🎨 NotY3dGenAI</div>'))
        display(HTML('<div class="subtitle">Professional AI-Powered 3D Model Generator</div>'))
        
        display(HTML(f"""
        <div class="info">
            <div style="display: flex; justify-content: space-between;">
                <div>
                    ✨ <b>Device:</b> {self.generator.device}<br>
                    🎯 <b>Status:</b> Ready
                </div>
                <div>
                    💾 <b>Storage:</b> Google Drive<br>
                    🎨 <b>Quality:</b> Professional
                </div>
            </div>
        </div>
        """))
        
        # Controls
        self.prompt = widgets.Textarea(
            value="Create a Male who is Profound, resonant deep tone that carries the weight of absolute authority and quiet, terrifying certainty",
            placeholder="Describe your 3D model in detail...",
            layout=widgets.Layout(width='100%', height='120px')
        )
        
        self.quality = widgets.Dropdown(
            options=['Ultra', 'High', 'Medium', 'Low'],
            value='High',
            description='Quality:',
            style={'description_width': 'initial'}
        )
        
        self.polygons = widgets.IntSlider(
            value=25000,
            min=10000,
            max=50000,
            step=5000,
            description='Polygons:',
            style={'description_width': 'initial'}
        )
        
        self.generate_btn = widgets.Button(
            description="🚀 GENERATE 3D MODEL",
            layout=widgets.Layout(width='100%', height='50px')
        )
        
        self.download_btn = widgets.Button(
            description="📥 DOWNLOAD MODEL",
            layout=widgets.Layout(width='100%', height='40px'),
            disabled=True
        )
        
        self.status = widgets.Output()
        
        controls = widgets.VBox([
            self.prompt,
            self.quality,
            self.polygons,
            self.generate_btn,
            self.download_btn,
            self.status
        ], layout=widgets.Layout(padding='10px'))
        
        self.viewer = widgets.Output()
        
        container = widgets.HBox([
            widgets.VBox([controls], layout=widgets.Layout(width='40%')),
            widgets.VBox([
                widgets.HTML("<h3 style='text-align:center; color:white;'>🎬 3D Viewer</h3>"),
                self.viewer
            ], layout=widgets.Layout(width='60%'))
        ])
        
        display(container)
        
        self.generate_btn.on_click(self.generate)
        self.download_btn.on_click(self.download)
        
        with self.viewer:
            self.show_placeholder()
    
    def show_placeholder(self):
        fig = go.Figure()
        fig.add_annotation(
            text="✨ Your 3D model will appear here ✨<br><br>Click GENERATE to start",
            x=0.5, y=0.5, showarrow=False,
            font=dict(size=16, color="gray")
        )
        fig.update_layout(
            height=500,
            paper_bgcolor='black',
            plot_bgcolor='black'
        )
        fig.show()
    
    def generate(self, btn):
        with self.status:
            clear_output()
            start_time = time.time()
            
            print("🎨 Generating professional 3D model...")
            print(f"📝 Prompt: {self.prompt.value[:80]}...")
            print(f"⚙️ Quality: {self.quality.value}")
            print("-" * 40)
            
            try:
                # Quality settings
                quality_map = {'Ultra': 1.2, 'High': 1.0, 'Medium': 0.7, 'Low': 0.5}
                detail = quality_map[self.quality.value]
                
                # Step 1: Create mesh
                print("\n🔺 Creating 3D mesh...")
                mesh = self.generator.create_human_mesh(self.prompt.value, detail)
                
                # Step 2: Simplify if needed
                if len(mesh.faces) > self.polygons.value:
                    print(f"   Simplifying: {len(mesh.faces)} -> {self.polygons.value} faces")
                    mesh = mesh.simplify_quadric_decimation(self.polygons.value)
                
                # Step 3: Create texture
                print("🎨 Creating texture...")
                texture = self.generator.create_authoritative_texture(self.prompt.value)
                
                # Step 4: Save model
                print("💾 Saving model...")
                timestamp = int(time.time())
                safe_prompt = "authoritative_man"
                filename = f"noty3d_{safe_prompt}_{timestamp}.obj"
                local_path = f"/content/noty3d_models/{filename}"
                
                # Export mesh
                mesh.export(local_path)
                
                # Save to Drive
                drive_path = f"/content/drive/MyDrive/NotY3D_Models/{filename}"
                import shutil
                shutil.copy(local_path, drive_path)
                
                self.current_model_path = local_path
                self.current_mesh = mesh
                
                # Calculate time
                elapsed = time.time() - start_time
                
                print("\n" + "=" * 40)
                print("✅ MODEL GENERATED SUCCESSFULLY!")
                print("=" * 40)
                print(f"⏱️ Time: {elapsed:.1f} seconds")
                print(f"🔺 Vertices: {len(mesh.vertices):,}")
                print(f"🔻 Faces: {len(mesh.faces):,}")
                print(f"💾 Saved: {local_path}")
                
                # Enable download button
                self.download_btn.disabled = False
                
                # Display in 3D viewer
                with self.viewer:
                    clear_output()
                    self.display_3d(mesh)
                
                print("\n🎉 Generation complete! Click DOWNLOAD to save your model.")
                
            except Exception as e:
                print(f"\n❌ Error: {str(e)}")
                import traceback
                traceback.print_exc()
    
    def display_3d(self, mesh):
        """Display 3D model with professional viewer"""
        try:
            vertices = mesh.vertices
            faces = mesh.faces
            
            # Sample for performance
            if len(vertices) > 10000:
                step = len(vertices) // 8000
                vertices = vertices[::step]
                faces = faces[::max(1, step//2)]
            
            # Create 3D visualization
            fig = go.Figure(data=[
                go.Mesh3d(
                    x=vertices[:, 0],
                    y=vertices[:, 1],
                    z=vertices[:, 2],
                    i=faces[:, 0],
                    j=faces[:, 1],
                    k=faces[:, 2],
                    intensity=vertices[:, 1],
                    colorscale='Viridis',
                    opacity=0.9,
                    lighting=dict(
                        ambient=0.6,
                        diffuse=0.8,
                        specular=0.5,
                        roughness=0.3
                    ),
                    lightposition=dict(x=200, y=200, z=400)
                )
            ])
            
            fig.update_layout(
                scene=dict(
                    camera=dict(
                        eye=dict(x=1.5, y=1.5, z=1.5),
                        up=dict(x=0, y=1, z=0)
                    ),
                    aspectmode='data',
                    bgcolor='#111111',
                    xaxis=dict(visible=False),
                    yaxis=dict(visible=False),
                    zaxis=dict(visible=False)
                ),
                width=700,
                height=500,
                margin=dict(l=0, r=0, b=0, t=0),
                paper_bgcolor='#111111'
            )
            
            fig.show()
            
        except Exception as e:
            print(f"⚠️ Viewer error: {e}")
    
    def download(self, btn):
        """Download the generated model"""
        if hasattr(self, 'current_model_path') and self.current_model_path and os.path.exists(self.current_model_path):
            print("📥 Downloading...")
            files.download(self.current_model_path)
            print("✅ Download started!")
        else:
            print("❌ No model available. Generate first.")

# Launch
print("""
╔════════════════════════════════════════════════════════════╗
║         🎨 NotY3dGenAI - Professional 3D Generator        ║
║                                                            ║
║  • Creates detailed human models                          ║
║  • Professional 3D viewer                                 ║
║  • One-click download                                     ║
║  • Auto-save to Drive                                     ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
""")

app = NotY3dGenAI()
print("\n✨ Ready! Click GENERATE to create your authoritative male figure.")